# Diversificando modelos de prediccion

En el notebook anterior de Machine Learning desarrollamos el procedimiento que utilizamos para crear un modelo de prediccion de partidos en este entorno, y creamos modelos con todos los tipos datos disponibles de la base de datos.   

El enfoque de utilizar todos los tipos de datos para estos primeros modelos nos limito a una muestra muy chica de partidos para entrenamiento. Por lo cual luego de una pequeña evaluacion de los modelos, nos planteamos la pregunta:  

¿Es mejor entrenar el modelo variables mas diversas y una muestra reducida, o utilizar menos variables pero tenes mas informacion de partidos?  

En este notebook vamos a desarrollar que modelos podemos crear con distintas disponibilidades de variables para verificar que tan bien llegan a predecir.  


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import empiricaldist
import sqlite3
import scipy.stats
import statsmodels.api as sm
conn = sqlite3.connect("../data/Datos_Limpios.db")

In [2]:
query = "SELECT * " \
"FROM world_cup_matches"
wcm = pd.read_sql_query(query, conn)
wcm

,match_id,date,home_team,home_team_id,away_team,away_team_id,round,home_score,away_score,neutral,...,away_xg,home_possession,away_possession,home_sot,away_sot,home_shots,away_shots,result,away_is_local,home_is_local
0,1408,1930-07-13,United States,175,Belgium,17,Group stage,3,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home team win,0,0
1,1409,1930-07-13,France,61,Mexico,110,Group stage,4,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home team win,0,0
2,1410,1930-07-14,Yugoslavia,144,Brazil,25,Group stage,2,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home team win,0,0
3,1411,1930-07-14,Romania,139,Peru,134,Group stage,3,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home team win,0,0
4,1412,1930-07-15,Argentina,7,France,61,Group stage,1,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home team win,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063,48061,2026-07-11,Norway,127,England,53,Quarter-finals,1,2,1,...,0.96,48.0,52.0,4.0,8.0,13.0,14.0,away team win,0,0
1064,48062,2026-07-14,France,61,Spain,155,Semi-finals,0,2,1,...,1.63,49.0,51.0,3.0,2.0,10.0,10.0,away team win,0,0
1065,48063,2026-07-15,England,53,Argentina,7,Semi-finals,1,2,1,...,1.80,36.0,64.0,2.0,5.0,5.0,15.0,away team win,0,0
1066,48064,2026-07-18,France,61,England,53,Third-place match,4,6,1,...,2.88,46.0,54.0,9.0,11.0,19.0,19.0,away team win,0,0


In [3]:
query = "SELECT * " \
"FROM eloratings_clean"
elo_ratings = pd.read_sql_query(query, conn)
elo_ratings

,date,team,rating,change,team_id
0,1872-11-30,England,2003,3,53.0
1,1872-11-30,Scotland,1997,-3,185.0
2,1873-03-08,England,2014,11,53.0
3,1873-03-08,Scotland,1986,-11,185.0
4,1874-03-07,England,2006,-8,53.0
...,...,...,...,...,...
6673,2025-12-13,Northern Mariana Islands,432,0,230.0
6674,2025-12-13,Cocos Islands,422,0,231.0
6675,2025-12-13,Palau,402,0,232.0
6676,2025-12-13,Eastern Samoa,389,0,214.0


In [4]:
query = "SELECT *" \
"FROM results_shootouts"

rs = pd.read_sql_query(query, conn)

rs["date"] = pd.to_datetime(rs["date"])
rs

,match_id,date,home_team,away_team,home_team_id,away_team_id,home_score,away_score,neutral,country,...,has_shootouts,winner_on_shootouts,winner,tournament,tournament_id,result,local_team_id,local,away_is_local,home_is_local
0,3204,1950-01-24,Zambia,DR Congo,183,194,1,0,0,Northern Rhodesia,...,0,NaN,Zambia,Friendly,97,home team win,183.0,home team,0,1
1,3205,1950-02-17,Egypt,Greece,51,67,2,0,0,Egypt,...,0,NaN,Egypt,Friendly,97,home team win,51.0,home team,0,1
2,3206,1950-02-26,Bolivia,Chile,22,36,2,0,0,Bolivia,...,0,NaN,Bolivia,Friendly,97,home team win,22.0,home team,0,1
3,3207,1950-02-26,Croatia,Serbia,42,144,4,1,0,Yugoslavia,...,0,NaN,Croatia,Friendly,97,home team win,144.0,away team,1,0
4,3208,1950-03-05,Italy,Belgium,84,17,3,1,0,Italy,...,0,NaN,Italy,Friendly,97,home team win,84.0,home team,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44857,48061,2026-07-11,Norway,England,127,53,1,2,1,United States,...,0,NaN,England,FIFA World Cup,1,away team win,NaN,NaN,0,0
44858,48062,2026-07-14,France,Spain,61,155,0,2,1,United States,...,0,NaN,Spain,FIFA World Cup,1,away team win,NaN,NaN,0,0
44859,48063,2026-07-15,England,Argentina,53,7,1,2,1,United States,...,0,NaN,Argentina,FIFA World Cup,1,away team win,NaN,NaN,0,0
44860,48064,2026-07-18,France,England,61,53,4,6,1,United States,...,0,NaN,England,FIFA World Cup,1,away team win,NaN,NaN,0,0


## Dimensionando los ejes temporales  

Para entender el tamanio de muestra que podemos utilizar para cada modelo, debemos recordar la disponibilidad de cada variable:  

- Elo: 1872-2025
- Momentum previo al mundial (estadisticas de results_shootouts rs): 1950-2026
- Rendimiento en el mundial (estadisticas de world_cup_matches wcm): 1930-2026


Para no tomar datos de un futbol muy antiguo, tomaremos datos desde 1998 (primera edicion con 32 selecciones), y hasta 2022, para que cada modelo se pruebe sobre el 2026.  

En el notebook previo evitamos utilizar la localidad como variable explicativa porque el modelo no podia estimarla correctamente y tomamos como posible factor el tamaño de la muestra, veamos si con el mayor tamaño de muestra de estos modelos podemos hacer convergir el modelo con esta variable.

En este notebook vamos a explicar menos como se consiguen los datos para cada modelo, ya que seria repetitivo y la logica ya se explaya en el notebook anterior.  

## Modelo 1: Elo

Tal como en el notebook previo, utilizaremos el modelo con solo elo que es una de las mejores variables predictivas como un baseline para comparar el desempeño de los demas modelos. Se entrenara con el elo de cada equipo de los partidos mundialistas desde 1998 a 2022.  


### Construyendo las variables y el dataset de entrenamiento
Empecemos consiguiendo el elo de cada equipo para cada partido del mundial:  
pasamos las fechas de cada tabla a formato DateTime de Pandas:

In [5]:
wcm["date"] = pd.to_datetime(wcm["date"])
elo_ratings["date"] = pd.to_datetime(elo_ratings["date"])

Establecemos el subconjunto valido para entrenar este modelo y reutilizamos la funcion del notebook previo que utilizamos para conseguir el elo de ambos equipos 

In [6]:
def conseguir_elo(home_team_id, away_team_id, fecha):
    elo_ratings_valido = elo_ratings[elo_ratings["date"] < fecha]

    elo_ratings_home = elo_ratings_valido[elo_ratings_valido["team_id"] == home_team_id]
    home_team_rating = elo_ratings_home.tail(1)["rating"].iloc[0]

    elo_ratings_away = elo_ratings_valido[elo_ratings_valido["team_id"] == away_team_id]
    away_team_rating = elo_ratings_away.tail(1)["rating"].iloc[0]

    return home_team_rating, away_team_rating

subconjunto_valido_1 = wcm[(wcm["date"].dt.year >= 1998)].copy()
subconjunto_valido_1["date"].min(), subconjunto_valido_1["date"].max()

(Timestamp('1998-06-10 00:00:00'), Timestamp('2026-07-19 00:00:00'))

Ahora construimos el dataset de entrenamiento de este modelo, y construimos los datos de cada partido del subconjunto valido para este:

In [7]:
dataset_entrenamiento_1 = pd.DataFrame(columns=("match_id", "date", "home_team", "away_team", "home_team_id",
                                        "away_team_id", "elo_diff", "home_local", "away_local", "result"))

for _, partido in subconjunto_valido_1.iterrows():
    fecha = partido["date"]
    home_team_id = partido["home_team_id"]
    away_team_id = partido["away_team_id"]

    home_elo, away_elo = conseguir_elo(home_team_id, away_team_id, fecha)

    dataset_entrenamiento_1.loc[len(dataset_entrenamiento_1)] = [
            partido["match_id"],
            fecha,
            partido["home_team"],
            partido["away_team"],
            home_team_id,
            away_team_id,

            home_elo - away_elo,

            partido["home_is_local"],
            partido["away_is_local"],

            partido["result"],
            ]

dataset_entrenamiento_1

,match_id,date,home_team,away_team,home_team_id,away_team_id,elo_diff,home_local,away_local,result
0,22393,1998-06-10,Brazil,Scotland,25,185,261,0,0,home team win
1,22394,1998-06-10,Morocco,Norway,114,127,-244,0,0,draw
2,22395,1998-06-11,Cameroon,Austria,31,10,-27,0,0,draw
3,22396,1998-06-11,Italy,Chile,84,36,267,0,0,draw
4,22397,1998-06-12,Paraguay,Bulgaria,133,27,26,0,0,draw
...,...,...,...,...,...,...,...,...,...,...
547,48061,2026-07-11,Norway,England,127,53,-120,0,0,away team win
548,48062,2026-07-14,France,Spain,61,155,-109,0,0,away team win
549,48063,2026-07-15,England,Argentina,53,7,-71,0,0,away team win
550,48064,2026-07-18,France,England,61,53,20,0,0,away team win


Dividimos en serie de entrenamiento y serie de prueba:

In [8]:
training_series_1 = dataset_entrenamiento_1[(dataset_entrenamiento_1["date"].dt.year < 2026)].copy()
test_series_1 = dataset_entrenamiento_1[(dataset_entrenamiento_1["date"].dt.year == 2026)].copy()
training_series_1["date"].min(), training_series_1["date"].max(), test_series_1["date"].min(), test_series_1["date"].max()

(Timestamp('1998-06-10 00:00:00'),
 Timestamp('2022-12-18 00:00:00'),
 Timestamp('2026-06-11 00:00:00'),
 Timestamp('2026-07-19 00:00:00'))

In [9]:
len(training_series_1), len(test_series_1)

(448, 104)

### Creando el modelo:
Ahora si, ya que tenemos los datos podemos construir el modelo:

In [10]:
X = training_series_1[["elo_diff", "home_local"]]
X = sm.add_constant(X) # Aniadimos constante para que sirva de intercept
y = training_series_1["result"]

M1 = sm.MNLogit(y, X)
resultado = M1.fit()

display(resultado.summary())

Optimization terminated successfully.
         Current function value: 0.939899
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                 result   No. Observations:                  448
Model:                        MNLogit   Df Residuals:                      442
Method:                           MLE   Df Model:                            4
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.08976
Time:                        16:34:37   Log-Likelihood:                -421.07
converged:                       True   LL-Null:                       -462.60
Covariance Type:            nonrobust   LLR p-value:                 3.936e-17
========================================================================================
         result=draw       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -0.5497      0.144     -3.807      0.000      -0.833      -0.267
elo_diff                 0.0025      0.001      3.441      0.001       0.001       0.004
home_local              -0.3080      0.603     -0.511      0.610      -1.490       0.874
----------------------------------------------------------------------------------------
result=home team win       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.0877      0.126      0.695      0.487      -0.160       0.335
elo_diff                 0.0050      0.001      7.904      0.000       0.004       0.006
home_local               0.9669      0.404      2.391      0.017       0.174       1.759
========================================================================================
"""

Este modelo con localidad pudo converger correctamente: Elo diff es estadisticamente significativo en ambas categorias, y home_local solo lo es en victoria de equipo local. Asi que ambas variables nos pueden ayudar a predecir.  
 Veamos como se ve en esta observacion la distribucion de valores de home_local con el resultado del partido:  

In [11]:
pd.crosstab(
    training_series_1["home_local"],
    training_series_1["result"],
    margins=True
)

result,away team win,draw,home team win,All
home_local,,,,
0,140,79,186,405
1,12,4,27,43
All,152,83,213,448


En este caso con mayor muestra, si se presentaron casos en los que un equipo siendo local empato, por lo que probablemente el modelo anterior no pudo converger debido a su limitada cantidad de observaciones.  

Enfocandonos en el modelo, su peudo R2 es 0.0897, y su Log-Likelihood es -421.07.  
No conviene comparar el valor de Log-Likelihood de estos modelos con el de los modelos anteriores, porque por la naturaleza de como se obtiene este valor (suma de logaritmo natural probabilidades), este tendra mayores magnitudes al aumentar su tamaño de muestra.  


## Modelo 2: Elo + partidos de results_shootouts
Este modelo sera particular por que no comparara partidos de mundiales, sino de results_shootouts, que recordemos que tiene partidos en general de futbol a nivel selecciones. Por lo que tendra una muestra mucho mayor al no estar solo limitado a partidos de mundial, pero tambien teniendolos. Cabe destacar que como estaria entrenado este modelo ponderaria igual cada partido ignorando la competicion, es decir ponderaria igual un amistoso que un partido de mundial por ejemplo.  
Por lo cual este modelo podria ser una mejor aproximacion a un modelo general de prediccion de partidos a nivel selecciones, pero por su gran tamaño de muestra puede comparar mas informacion que los demas. El eje temporal de este modelo sera de 1998 a 2026  


### Construyendo las variables y el dataset de entrenamiento

Reutilizamos la funcion del notebook anterior que nos permitio obtener datos de results_shootouts para cada equipo.  

In [12]:
def construir_variables_partidos_previos(rs_sub, team_id):

    rs_sub = rs_sub.copy()

    rs_sub["posicion"] = np.where(
        rs_sub["home_team_id"] == team_id,
        "home_team",
        "away_team"
    )

    rs_sub["goles_anotados"] = np.where(
        rs_sub["posicion"] == "home_team",
        rs_sub["home_score"],
        rs_sub["away_score"]
    )

    rs_sub["goles_recibidos"] = np.where(
        rs_sub["posicion"] == "home_team",
        rs_sub["away_score"],
        rs_sub["home_score"]
    )


    rs_sub["victoria"] = (
        rs_sub["goles_anotados"] > rs_sub["goles_recibidos"]
    ).astype(int)

    rs_sub["derrota"] = (
        rs_sub["goles_anotados"] < rs_sub["goles_recibidos"]
    ).astype(int)

    avg_goles_anotados = rs_sub["goles_anotados"].mean()
    avg_goles_recibidos = rs_sub["goles_recibidos"].mean()
    avg_victorias = rs_sub["victoria"].mean()
    avg_derrotas = rs_sub["derrota"].mean()

    return (
        avg_goles_anotados,
        avg_goles_recibidos,
        avg_victorias,
        avg_derrotas
    )


Obtengamos el subconjunto valido de este modelo: Tomemos como subconjunto valido para este modelo solo partidos no amistosos donde ambos equipos hayan disputado alguna vez un mundial:  
Los partidos de results_shootouts no amistosos son tales que el tournament_id de ese partido no es igual a 97, que es el id de partido amistoso:

In [13]:
rs[rs["tournament_id"] == 97]["tournament"].unique()

<StringArray>
['Friendly']
Length: 1, dtype: str

In [14]:

disputaron_mundial = pd.concat([
    wcm["home_team_id"],
    wcm["away_team_id"]
    ], ignore_index=True).unique()
disputaron_mundial

array([175,  61, 144, 139,   7,  36, 176, 133,  25, 159,  10,  65, 155,
        45,  76,  84, 160,  43, 110,  53,  22,  17, 185, 153,  38, 140,
        27, 124, 137,  83, 114,  52, 194, 136,  73, 119,   9,  80, 134,
         3,  74,  32, 126,  81,  46, 174,  41,  31,  82, 127, 123, 142,
        67,  86, 152,  87,  42,  37, 169, 143, 149, 170,  50, 168,   5,
       166,  85,  66, 173, 120, 148,  23,  51,  77, 131, 138, 181, 177,
        88, 186,  33,  79,  91])

In [15]:

subconjunto_valido_2 = rs[
    (rs["tournament_id"] != 97) &
     (rs["date"].dt.year >= 1998) &
    (
        rs["home_team_id"].isin(disputaron_mundial) &
        rs["away_team_id"].isin(disputaron_mundial)
    )
].copy()

subconjunto_valido_2


,match_id,date,home_team,away_team,home_team_id,away_team_id,home_score,away_score,neutral,country,...,has_shootouts,winner_on_shootouts,winner,tournament,tournament_id,result,local_team_id,local,away_is_local,home_is_local
18919,22123,1998-01-27,Egypt,South Korea,51,153,0,2,1,Thailand,...,0,NaN,South Korea,King's Cup,80,away team win,NaN,NaN,0,0
18922,22126,1998-01-28,Iran,Nigeria,80,123,0,1,1,Hong Kong,...,0,NaN,Nigeria,Lunar New Year Cup,128,away team win,NaN,NaN,0,0
18930,22134,1998-01-31,Iran,Chile,80,36,1,1,1,Hong Kong,...,1,Iran,Iran,Lunar New Year Cup,128,home team win,NaN,NaN,0,0
18931,22135,1998-01-31,Egypt,South Korea,51,153,1,1,1,Thailand,...,1,South Korea,South Korea,King's Cup,80,away team win,NaN,NaN,0,0
18935,22139,1998-02-01,United States,Cuba,175,43,3,0,0,United States,...,0,NaN,United States,Gold Cup,40,home team win,175.0,home team,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44857,48061,2026-07-11,Norway,England,127,53,1,2,1,United States,...,0,NaN,England,FIFA World Cup,1,away team win,NaN,NaN,0,0
44858,48062,2026-07-14,France,Spain,61,155,0,2,1,United States,...,0,NaN,Spain,FIFA World Cup,1,away team win,NaN,NaN,0,0
44859,48063,2026-07-15,England,Argentina,53,7,1,2,1,United States,...,0,NaN,Argentina,FIFA World Cup,1,away team win,NaN,NaN,0,0
44860,48064,2026-07-18,France,England,61,53,4,6,1,United States,...,0,NaN,England,FIFA World Cup,1,away team win,NaN,NaN,0,0


Construimos el dataset de entrenamiento:

In [16]:
dataset_entrenamiento_2 = pd.DataFrame(columns=("match_id", "date", "home_team", "away_team",
     "home_team_id", "away_team_id", "elo_diff", "home_local", "away_local",
     "goles_anotados_diff_rs", "goles_recibidos_diff_rs", "victorias_diff_rs", "derrotas_diff_rs",
     "result"))

for _, partido in subconjunto_valido_2.iterrows():
    fecha = partido["date"]
    home_team_id = partido["home_team_id"]
    away_team_id = partido["away_team_id"]

    home_team_matches_rs = rs[
    (rs["date"] < fecha) &
    (
        (rs["home_team_id"] == home_team_id) |
        (rs["away_team_id"] == home_team_id)
    )].tail(10)

    away_team_matches_rs = rs[
    (rs["date"] < fecha) &
    (
        (rs["home_team_id"] == away_team_id) |
        (rs["away_team_id"] == away_team_id)
    )
    ].tail(10)

    stats_home_rs = construir_variables_partidos_previos(home_team_matches_rs, home_team_id)
    stats_away_rs = construir_variables_partidos_previos(away_team_matches_rs, away_team_id)
    home_elo, away_elo = conseguir_elo(home_team_id, away_team_id, fecha)

    dataset_entrenamiento_2.loc[len(dataset_entrenamiento_2)] = [
            partido["match_id"],
            fecha,
            partido["home_team"],
            partido["away_team"],
            home_team_id,
            away_team_id,

            home_elo - away_elo,

            partido["home_is_local"],
            partido["away_is_local"],

            stats_home_rs[0] - stats_away_rs[0], # goles anotados rs diff
            stats_home_rs[1] - stats_away_rs[1], # goles recibidos rs diff
            stats_home_rs[2] - stats_away_rs[2], # victorias rs diff
            stats_home_rs[3] - stats_away_rs[3], # derrotas rs diff
            
            partido["result"]

            ]

In [17]:
dataset_entrenamiento_2["date"].min(), dataset_entrenamiento_2["date"].max()

(Timestamp('1998-01-27 00:00:00'), Timestamp('2026-07-19 00:00:00'))

Y dividimos el conjunto de prueba y de entrenamiento entre los partidos antes y despues del mundial 2026:

In [18]:
fecha_inicio = wcm[wcm["date"].dt.year == 2026].head(1)["date"].iloc[0]
fecha_inicio

Timestamp('2026-06-11 00:00:00')

In [19]:
training_series_2 = dataset_entrenamiento_2[dataset_entrenamiento_2["date"] < fecha_inicio]
test_series_2 = dataset_entrenamiento_2[dataset_entrenamiento_2["date"] >= fecha_inicio]

training_series_2["date"].min(), training_series_2["date"].max(), test_series_2["date"].min(), test_series_2["date"].min()

(Timestamp('1998-01-27 00:00:00'),
 Timestamp('2026-05-30 00:00:00'),
 Timestamp('2026-06-11 00:00:00'),
 Timestamp('2026-06-11 00:00:00'))

In [20]:
len(training_series_2), len(test_series_2)

(5323, 104)

### Creando el modelo  


In [21]:
X = training_series_2[["elo_diff", "home_local", "goles_anotados_diff_rs",
                        "goles_recibidos_diff_rs",
                        "victorias_diff_rs", "derrotas_diff_rs"]]
X = sm.add_constant(X) # Aniadimos constante para que sirva de intercept
y = training_series_2["result"]

M2 = sm.MNLogit(y, X)
resultado = M2.fit()

display(resultado.summary())

Optimization terminated successfully.
         Current function value: 0.963315
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                 result   No. Observations:                 5323
Model:                        MNLogit   Df Residuals:                     5309
Method:                           MLE   Df Model:                           12
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.08795
Time:                        16:35:52   Log-Likelihood:                -5127.7
converged:                       True   LL-Null:                       -5622.2
Covariance Type:            nonrobust   LLR p-value:                4.330e-204
===========================================================================================
            result=draw       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.5730      0.067     -8.532      0.000      -0.705      -0.441
elo_diff                    0.0023      0.000      9.849      0.000       0.002       0.003
home_local                  0.7597      0.084      9.086      0.000       0.596       0.924
goles_anotados_diff_rs      0.0951      0.068      1.408      0.159      -0.037       0.228
goles_recibidos_diff_rs    -0.3179      0.105     -3.039      0.002      -0.523      -0.113
victorias_diff_rs          -0.1750      0.263     -0.664      0.507      -0.691       0.341
derrotas_diff_rs            0.2161      0.291      0.744      0.457      -0.353       0.786
-------------------------------------------------------------------------------------------
   result=home team win       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.0478      0.059     -0.810      0.418      -0.164       0.068
elo_diff                    0.0046      0.000     20.780      0.000       0.004       0.005
home_local                  0.8010      0.075     10.692      0.000       0.654       0.948
goles_anotados_diff_rs      0.2319      0.063      3.702      0.000       0.109       0.355
goles_recibidos_diff_rs    -0.3920      0.096     -4.075      0.000      -0.581      -0.203
victorias_diff_rs          -0.1661      0.241     -0.689      0.491      -0.638       0.306
derrotas_diff_rs            0.1681      0.266      0.632      0.527      -0.353       0.690
===========================================================================================
"""

El valor pseudo R2 de este modelo es 0.08795 y el Log-Likelihood es -5127 a comparacion del de modelo nulo (LLN) que es -5622.   
Elo_diff, home_local, goles_anotados_diff_rs y goles_recibidos_diff_rs son estadisticamente significativos en ambas ecuaciones. Excepto goles_anotados_diff_rs que no es significativo en empate.  
victorias_diff_rs y derrotas_diff no son significativos en ningun resultado, y ademas su error estandar es considerablemente alto en comparacion a las otras variables.  Veamos que pasa si le sacamos estas ultimas 2 variables al modelo:

In [22]:
X = training_series_2[["elo_diff", "home_local", "goles_anotados_diff_rs",
                        "goles_recibidos_diff_rs"]]
X = sm.add_constant(X) # Aniadimos constante para que sirva de intercept
y = training_series_2["result"]

M2 = sm.MNLogit(y, X)
resultado = M2.fit()

display(resultado.summary())

Optimization terminated successfully.
         Current function value: 0.963457
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                 result   No. Observations:                 5323
Model:                        MNLogit   Df Residuals:                     5313
Method:                           MLE   Df Model:                            8
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                 0.08782
Time:                        16:35:52   Log-Likelihood:                -5128.5
converged:                       True   LL-Null:                       -5622.2
Covariance Type:            nonrobust   LLR p-value:                7.501e-208
===========================================================================================
            result=draw       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.5754      0.067     -8.573      0.000      -0.707      -0.444
elo_diff                    0.0023      0.000      9.829      0.000       0.002       0.003
home_local                  0.7626      0.084      9.128      0.000       0.599       0.926
goles_anotados_diff_rs      0.0508      0.052      0.973      0.331      -0.052       0.153
goles_recibidos_diff_rs    -0.2349      0.072     -3.271      0.001      -0.376      -0.094
-------------------------------------------------------------------------------------------
   result=home team win       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.0497      0.059     -0.843      0.399      -0.165       0.066
elo_diff                    0.0046      0.000     20.976      0.000       0.004       0.005
home_local                  0.8032      0.075     10.728      0.000       0.656       0.950
goles_anotados_diff_rs      0.1925      0.048      4.004      0.000       0.098       0.287
goles_recibidos_diff_rs    -0.3219      0.066     -4.893      0.000      -0.451      -0.193
===========================================================================================
"""

Con esta version del modelo con 2 variables menos, el Log-Likelihood y el pseudo R2 bajan muy levemente.  (Pseudo R2 de 0.8795 a 0.8782 y Log-Likelihood de -5127 a -5128).  
Un cambio muy leve considerando que eliminamos 4 parametros en total, 2 variables para cada modelo. Ademas, este modelo es todavia mas limpio, con errores estandar todavia muy bajos y manteniendo la significancia estadistica igual en las demas variables.  
Con esto podemos concluir que victorias_diff_rs y derrotas_diff_rs no ayuda en el ajuste, por lo menos para el conjunto de entrenamiento.  

Nos quedaremos con esta version del modelo reducido.  


## Modelo 3: Elo + Rendimiento previo al mundial (estadisticas de results_shootouts) + Rendimiento en el mundial (estadisticas de world_cup_matches wcm). 
Juntaria datos igual que el modelo que hicimos en el notebook previo, con la diferencia que no tendria datos de sot, shots, xg y possession. Al igual que el modelo del notebook previo, como los valores en wcm seran NaN para los primeros partidos de cada mundial, estos deberan ser filtrados y no utilizados en este modelo. Este modelo se hara con datos de 1998 a 2026.  

### Construyendo las variables y el dataset de entrenamiento  
Devuelta, para este modelo usaremos una funcion del notebook pasado para obtener estadisticas de partidos de wcm y rs, pero en este caso no utilizaremos estadisticas del desarrollo del partido para wcm.

In [23]:
def construir_variables_partidos_mundial(wcm_sub, team_id):

    wcm_sub = wcm_sub.copy()

    wcm_sub["posicion"] = np.where(
        wcm_sub["home_team_id"] == team_id,
        "home_team",
        "away_team"
    )

    wcm_sub["goles_anotados"] = np.where(
        wcm_sub["posicion"] == "home_team",
        wcm_sub["home_score"],
        wcm_sub["away_score"]
    )

    wcm_sub["goles_recibidos"] = np.where(
        wcm_sub["posicion"] == "home_team",
        wcm_sub["away_score"],
        wcm_sub["home_score"]
    )

    wcm_sub["victoria"] = (
        wcm_sub["goles_anotados"] > wcm_sub["goles_recibidos"]
    ).astype(int)

    wcm_sub["derrota"] = (
        wcm_sub["goles_anotados"] < wcm_sub["goles_recibidos"]
    ).astype(int)

    avg_goles_anotados = wcm_sub["goles_anotados"].mean()
    avg_goles_recibidos = wcm_sub["goles_recibidos"].mean()

    avg_victorias = wcm_sub["victoria"].mean()
    avg_derrotas = wcm_sub["derrota"].mean()

    return (
        avg_goles_anotados,
        avg_goles_recibidos,
        avg_victorias,
        avg_derrotas,
    )

def construir_variables_rs(rs_sub, team_id):

    rs_sub = rs_sub.copy()

    rs_sub["posicion"] = np.where(
        rs_sub["home_team_id"] == team_id,
        "home_team",
        "away_team"
    )

    rs_sub["goles_anotados"] = np.where(
        rs_sub["posicion"] == "home_team",
        rs_sub["home_score"],
        rs_sub["away_score"]
    )

    rs_sub["goles_recibidos"] = np.where(
        rs_sub["posicion"] == "home_team",
        rs_sub["away_score"],
        rs_sub["home_score"]
    )


    rs_sub["victoria"] = (
        rs_sub["goles_anotados"] > rs_sub["goles_recibidos"]
    ).astype(int)

    rs_sub["derrota"] = (
        rs_sub["goles_anotados"] < rs_sub["goles_recibidos"]
    ).astype(int)

    avg_goles_anotados = rs_sub["goles_anotados"].mean()
    avg_goles_recibidos = rs_sub["goles_recibidos"].mean()
    avg_victorias = rs_sub["victoria"].mean()
    avg_derrotas = rs_sub["derrota"].mean()

    return (
        avg_goles_anotados,
        avg_goles_recibidos,
        avg_victorias,
        avg_derrotas
    )

El subconjunto valido seria el mismo que el del modelo 1, partidos de mundiales desde 1994 hasta 2026:

In [24]:
subconjunto_valido_3 = wcm[(wcm["date"].dt.year >= 1998)].copy()
subconjunto_valido_3["date"].min(), subconjunto_valido_3["date"].max()

(Timestamp('1998-06-10 00:00:00'), Timestamp('2026-07-19 00:00:00'))

In [25]:
dataset_entrenamiento_3 = pd.DataFrame(columns=("match_id", "date", "home_team", "away_team", "home_team_id", "away_team_id", "elo_diff", "goles_anotados_diff_wcm", "goles_recibidos_diff_wcm",
              "victorias_diff_wcm", "derrotas_diff_wcm", "home_local", "away_local",
             "goles_anotados_diff_rs", "goles_recibidos_diff_rs", "victorias_diff_rs", "derrotas_diff_rs",
             "result"))

for _, partido in subconjunto_valido_3.iterrows():
    fecha = partido["date"]
    home_team_id = partido["home_team_id"]
    away_team_id = partido["away_team_id"]

    home_team_matches_wcm = wcm[
        (wcm["date"].dt.year == fecha.year) &
        (wcm["date"] < fecha) &
        (
            (wcm["home_team_id"] == home_team_id) |
            (wcm["away_team_id"] == home_team_id)
        )]
        
    away_team_matches_wcm = wcm[
        (wcm["date"].dt.year == fecha.year) &
        (wcm["date"] < fecha) &
        (
            (wcm["home_team_id"] == away_team_id) |
            (wcm["away_team_id"] == away_team_id)
        )]
    
    home_team_matches_rs = rs[
        (rs["date"] < fecha) &
        (rs["tournament_id"] != 1) &
        (
            (rs["home_team_id"] == home_team_id) |
            (rs["away_team_id"] == home_team_id)
        )].tail(10)
    
    away_team_matches_rs = rs[
        (rs["date"] < fecha) &
        (rs["tournament_id"] != 1) &
        (
            (rs["home_team_id"] == away_team_id) |
            (rs["away_team_id"] == away_team_id)
        )
        ].tail(10)

    stats_home_rs = construir_variables_rs(home_team_matches_rs, home_team_id)
    stats_away_rs = construir_variables_rs(away_team_matches_rs, away_team_id)
    
    stats_home_wcm = construir_variables_partidos_mundial(home_team_matches_wcm, home_team_id)
    stats_away_wcm = construir_variables_partidos_mundial(away_team_matches_wcm, away_team_id)

    home_elo, away_elo = conseguir_elo(home_team_id, away_team_id, fecha)

    dataset_entrenamiento_3.loc[len(dataset_entrenamiento_3)] = [
            partido["match_id"],
            fecha,
            partido["home_team"],
            partido["away_team"],
            home_team_id,
            away_team_id,

            home_elo - away_elo,

            stats_home_wcm[0] - stats_away_wcm[0], #goles anotados wcm diff
            stats_home_wcm[1] - stats_away_wcm[1], #goles recibidos wcm diff
            stats_home_wcm[2] - stats_away_wcm[2], #victorias wcm diff 
            stats_home_wcm[3] - stats_away_wcm[3], #derrotas wcm diff

            partido["home_is_local"],
            partido["away_is_local"],

            stats_home_rs[0] - stats_away_rs[0], # goles anotados rs diff
            stats_home_rs[1] - stats_away_rs[1], # goles recibidos rs diff
            stats_home_rs[2] - stats_away_rs[2], # victorias rs diff
            stats_home_rs[3] - stats_away_rs[3], # derrotas rs diff

            partido["result"],
            ]

dataset_entrenamiento_3

,match_id,date,home_team,away_team,home_team_id,away_team_id,elo_diff,goles_anotados_diff_wcm,goles_recibidos_diff_wcm,victorias_diff_wcm,derrotas_diff_wcm,home_local,away_local,goles_anotados_diff_rs,goles_recibidos_diff_rs,victorias_diff_rs,derrotas_diff_rs,result
0,22393,1998-06-10,Brazil,Scotland,25,185,261,NaN,NaN,NaN,NaN,0,0,0.6,-0.6,0.2,-0.2,home team win
1,22394,1998-06-10,Morocco,Norway,114,127,-244,NaN,NaN,NaN,NaN,0,0,-1.4,0.3,-0.3,0.3,draw
2,22395,1998-06-11,Cameroon,Austria,31,10,-27,NaN,NaN,NaN,NaN,0,0,-1.2,-0.2,-0.2,0.0,draw
3,22396,1998-06-11,Italy,Chile,84,36,267,NaN,NaN,NaN,NaN,0,0,0.0,0.1,-0.1,0.1,draw
4,22397,1998-06-12,Paraguay,Bulgaria,133,27,26,NaN,NaN,NaN,NaN,0,0,-0.6,0.8,-0.6,0.2,draw
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,48061,2026-07-11,Norway,England,127,53,-120,0.200000,0.800000,0.000000,0.2,0,0,0.7,0.6,-0.2,0.0,away team win
548,48062,2026-07-14,France,Spain,61,155,-109,0.833333,0.166667,0.166667,0.0,0,0,-0.3,0.5,0.1,0.1,away team win
549,48063,2026-07-15,England,Argentina,53,7,-71,-0.666667,0.000000,-0.166667,0.0,0,0,-0.1,-0.1,0.0,0.0,away team win
550,48064,2026-07-18,France,England,61,53,20,0.285714,-0.571429,0.142857,0.0,0,0,0.1,0.7,0.0,0.0,away team win


Al igual que en el notebook pasado, este dataset tendra filas con valores NaN en los primeros partidos de cada equipo en cada mundial, porque wcm no tendra partidos para crear sus variables.  
Por eso eliminaremos las filas con valores NaN para poder usar el modelo:  


In [26]:
dataset_entrenamiento_3 = dataset_entrenamiento_3.dropna()
dataset_entrenamiento_3

,match_id,date,home_team,away_team,home_team_id,away_team_id,elo_diff,goles_anotados_diff_wcm,goles_recibidos_diff_wcm,victorias_diff_wcm,derrotas_diff_wcm,home_local,away_local,goles_anotados_diff_rs,goles_recibidos_diff_rs,victorias_diff_rs,derrotas_diff_rs,result
16,22410,1998-06-16,Brazil,Morocco,25,114,498,0.000000,-1.000000,1.000000,0.0,0,0,0.4,-0.6,0.2,-0.1,home team win
17,22411,1998-06-16,Scotland,Norway,185,127,-7,-1.000000,0.000000,0.000000,1.0,0,0,-1.6,0.3,-0.3,0.4,draw
18,22412,1998-06-17,Chile,Austria,36,10,34,1.000000,1.000000,0.000000,0.0,0,0,-1.1,0.1,-0.4,-0.1,draw
19,22413,1998-06-17,Italy,Cameroon,84,31,328,1.000000,1.000000,0.000000,0.0,0,0,0.1,0.4,-0.3,0.0,home team win
20,22414,1998-06-18,South Africa,Denmark,152,46,-240,-1.000000,3.000000,-1.000000,1.0,0,0,0.1,0.1,-0.1,-0.1,draw
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,48061,2026-07-11,Norway,England,127,53,-120,0.200000,0.800000,0.000000,0.2,0,0,0.7,0.6,-0.2,0.0,away team win
548,48062,2026-07-14,France,Spain,61,155,-109,0.833333,0.166667,0.166667,0.0,0,0,-0.3,0.5,0.1,0.1,away team win
549,48063,2026-07-15,England,Argentina,53,7,-71,-0.666667,0.000000,-0.166667,0.0,0,0,-0.1,-0.1,0.0,0.0,away team win
550,48064,2026-07-18,France,England,61,53,20,0.285714,-0.571429,0.142857,0.0,0,0,0.1,0.7,0.0,0.0,away team win


Luego hacemos la division de serie de prueba y serie de entrenamiento:


In [27]:
training_series_3 = dataset_entrenamiento_3[(dataset_entrenamiento_3["date"].dt.year < 2026)].copy()
test_series_3 = dataset_entrenamiento_3[(dataset_entrenamiento_3["date"].dt.year == 2026)].copy()

training_series_3["date"].min(), training_series_3["date"].max(), test_series_3["date"].min(), test_series_3["date"].max()

(Timestamp('1998-06-16 00:00:00'),
 Timestamp('2022-12-18 00:00:00'),
 Timestamp('2026-06-18 00:00:00'),
 Timestamp('2026-07-19 00:00:00'))

In [28]:
len(training_series_3), len(test_series_3)

(336, 80)

### Creando el modelo

In [29]:
X = training_series_3[["elo_diff", "home_local",
 "goles_anotados_diff_wcm", "goles_recibidos_diff_wcm",
 "victorias_diff_wcm", "derrotas_diff_wcm",
 "goles_anotados_diff_rs", "goles_recibidos_diff_rs",
 "victorias_diff_rs", "derrotas_diff_rs"]]
X = sm.add_constant(X) # Aniadimos constante para que sirva de intercept
y = training_series_3["result"]

M3 = sm.MNLogit(y, X)
resultado = M3.fit()

display(resultado.summary())

Optimization terminated successfully.
         Current function value: 0.880391
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                 result   No. Observations:                  336
Model:                        MNLogit   Df Residuals:                      314
Method:                           MLE   Df Model:                           20
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                  0.1339
Time:                        16:36:05   Log-Likelihood:                -295.81
converged:                       True   LL-Null:                       -341.54
Covariance Type:            nonrobust   LLR p-value:                 4.115e-11
============================================================================================
             result=draw       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -0.7370      0.193     -3.826      0.000      -1.115      -0.359
elo_diff                     0.0006      0.001      0.567      0.571      -0.002       0.003
home_local                  -0.9514      0.812     -1.171      0.242      -2.544       0.641
goles_anotados_diff_wcm      0.4374      0.183      2.385      0.017       0.078       0.797
goles_recibidos_diff_wcm    -0.2782      0.181     -1.535      0.125      -0.633       0.077
victorias_diff_wcm          -0.4917      0.504     -0.976      0.329      -1.479       0.496
derrotas_diff_wcm            0.1771      0.529      0.335      0.738      -0.860       1.214
goles_anotados_diff_rs       0.2707      0.325      0.832      0.405      -0.367       0.908
goles_recibidos_diff_rs     -0.6967      0.556     -1.253      0.210      -1.787       0.393
victorias_diff_rs           -0.6272      1.221     -0.514      0.607      -3.020       1.766
derrotas_diff_rs            -1.5897      1.522     -1.045      0.296      -4.573       1.393
--------------------------------------------------------------------------------------------
    result=home team win       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                        0.1459      0.153      0.956      0.339      -0.153       0.445
elo_diff                     0.0024      0.001      2.696      0.007       0.001       0.004
home_local                   0.6434      0.450      1.430      0.153      -0.238       1.525
goles_anotados_diff_wcm      0.1547      0.154      1.008      0.314      -0.146       0.456
goles_recibidos_diff_wcm    -0.1443      0.154     -0.940      0.347      -0.445       0.157
victorias_diff_wcm           0.2851      0.405      0.704      0.481      -0.509       1.079
derrotas_diff_wcm            0.3186      0.433      0.736      0.461      -0.529       1.167
goles_anotados_diff_rs       0.4054      0.254      1.596      0.111      -0.093       0.903
goles_recibidos_diff_rs     -0.0217      0.425     -0.051      0.959      -0.854       0.811
victorias_diff_rs           -0.6559      0.958     -0.685      0.494      -2.534       1.222
derrotas_diff_rs            -2.6241      1.176     -2.231      0.026      -4.929      -0.319
============================================================================================
"""

Este modelo convergio con un pseudo R2 de 0.133 y Log-Likelihood de -295.8 sobre LL-Null de -341.5.  
Dentro de los que hicimos es el que mejor se ajusta a la serie de entrenamiento.  

En empate, goles_anotados_diff_wcm tiene coeficiente positivo (0.043) y es estadisticamente significativo (p = 0.017), Es decir que hay una asociacion entre mayor diferencia de goles anotados del local frente a visitante y empate por sobre victoria visitante. 
 En victoria local elo_diff tiene significancia estadistica junto con derrotas_diff_rs con gran coeficiente negativo (-2.62) y bajo valor p (0.026), es decir que hay una asociacion entre mayor promedio de derrotas del equipo visitante (es decir derrotas_diff_rs negativo) por sobre el local antes del mundial, y victoria local.  

 Evaluemos una version reducida y parsimonica de M3, sacando las variables que no parecen aportar informacion util para las predicciones:

In [30]:
X = training_series_3[["elo_diff", "home_local",
 "goles_anotados_diff_wcm", "derrotas_diff_rs"]]
X = sm.add_constant(X) # Aniadimos constante para que sirva de intercept
y = training_series_3["result"]

M3 = sm.MNLogit(y, X)
resultado = M3.fit()

display(resultado.summary())

Optimization terminated successfully.
         Current function value: 0.896689
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                 result   No. Observations:                  336
Model:                        MNLogit   Df Residuals:                      326
Method:                           MLE   Df Model:                            8
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                  0.1179
Time:                        16:36:05   Log-Likelihood:                -301.29
converged:                       True   LL-Null:                       -341.54
Covariance Type:            nonrobust   LLR p-value:                 3.865e-14
===========================================================================================
            result=draw       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.7283      0.185     -3.936      0.000      -1.091      -0.366
elo_diff                    0.0010      0.001      0.976      0.329      -0.001       0.003
home_local                 -0.9701      0.811     -1.196      0.232      -2.560       0.620
goles_anotados_diff_wcm     0.3234      0.130      2.492      0.013       0.069       0.578
derrotas_diff_rs           -2.7689      0.989     -2.799      0.005      -4.708      -0.830
-------------------------------------------------------------------------------------------
   result=home team win       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                       0.1614      0.146      1.105      0.269      -0.125       0.448
elo_diff                    0.0030      0.001      3.647      0.000       0.001       0.005
home_local                  0.6514      0.446      1.459      0.144      -0.223       1.526
goles_anotados_diff_wcm     0.1976      0.106      1.857      0.063      -0.011       0.406
derrotas_diff_rs           -2.6020      0.785     -3.313      0.001      -4.141      -1.063
===========================================================================================
"""

Esta version reducida del modelo con 4 variables replica considerablemente los resultados del modelo completo con 10 variables, y es considerablemente mas limpio y estable.   
La mayoria de las variables es estadisticamente significativa en por lo menos una de las dos ecuaciones y la perdida de ajuste sobre la serie de entrenamiento (Pseudo R2 de 0.1339 a 0.1179  y Log-Likelihood de -295.8 a -301.2) no es muy alta considerando la cantidad de variables sacadas del modelo.  

## Evaluando los modelos

Con esto ya podemos ir comparando cada modelo con su version de su serie de entrenamiento, y luego con una serie de entrenamiento en comun entre todos los modelos para poder comparar su rendimiento en conjunto.  
Ajustamos los modelos:  

In [31]:
resultado_M1 = M1.fit()

resultado_M2 = M2.fit()

resultado_M3 = M3.fit()

Optimization terminated successfully.
         Current function value: 0.939899
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.963457
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.896689
         Iterations 6


Preparamos las series de prueba para que puedan ser predecidas por los modelos que utilizamos:  

In [32]:
X_test_1 = test_series_1[["elo_diff", "home_local"]]
X_test_1 = sm.add_constant(X_test_1)

X_test_2 = test_series_2[["elo_diff", "home_local", "goles_anotados_diff_rs", 
                         "goles_recibidos_diff_rs"]]
X_test_2 = sm.add_constant(X_test_2)

X_test_3 = test_series_3[["elo_diff", "home_local", "goles_anotados_diff_wcm",
                          "derrotas_diff_rs"]]
X_test_3 = sm.add_constant(X_test_3)

Les brindamos ahora a los resultados de los modelos las series de entrenamiento:

In [33]:
probabilidades_M1 = resultado_M1.predict(X_test_1)
probabilidades_M2 = resultado_M2.predict(X_test_2)
probabilidades_M3 = resultado_M3.predict(X_test_3)

display(probabilidades_M1)

,0,1,2
448,0.324631,0.213507,0.461863
449,0.066601,0.059789,0.873610
450,0.092106,0.069061,0.838833
451,0.311447,0.106846,0.581706
452,0.479153,0.212899,0.307948
...,...,...,...
547,0.492974,0.211605,0.295420
548,0.482119,0.212638,0.305243
549,0.444457,0.215289,0.340254
550,0.355532,0.215551,0.428917


Para visualizar mejor los resultados de estas predicciones, podemos reutilizar la funcion del notebook anterior:

In [34]:
def obtener_predicciones(modelo, probabilidades, test):
    predicciones = test[
        ["match_id", "date", "home_team", "away_team", "result"]
    ].copy()

    predicciones["p_away"] = probabilidades[0].values
    predicciones["p_draw"] = probabilidades[1].values
    predicciones["p_home"] = probabilidades[2].values

    predicciones["prediccion"] = (
        probabilidades.idxmax(axis=1)
        .map(modelo.model._ynames_map)
        .values
    )

    predicciones["correcta"] = (
        predicciones["prediccion"] == predicciones["result"]
    )

    return predicciones

Asi obtenemos para cada prediccion un dataframe con los datos que nos interesan de esta prediccion:

In [35]:
predicciones_M1 = obtener_predicciones(resultado_M1, probabilidades_M1, test_series_1)

predicciones_M2 = obtener_predicciones(resultado_M2, probabilidades_M2, test_series_2)

predicciones_M3 = obtener_predicciones(resultado_M3, probabilidades_M3, test_series_3)

In [36]:
predicciones_M1

,match_id,date,home_team,away_team,result,p_away,p_draw,p_home,prediccion,correcta
448,47962,2026-06-11,South Korea,Czech Republic,home team win,0.324631,0.213507,0.461863,home team win,True
449,47963,2026-06-11,Mexico,South Africa,home team win,0.066601,0.059789,0.873610,home team win,True
450,47964,2026-06-12,Canada,Bosnia and Herzegovina,draw,0.092106,0.069061,0.838833,home team win,False
451,47965,2026-06-12,United States,Paraguay,home team win,0.311447,0.106846,0.581706,home team win,True
452,47966,2026-06-13,Australia,Turkey,home team win,0.479153,0.212899,0.307948,away team win,False
...,...,...,...,...,...,...,...,...,...,...
547,48061,2026-07-11,Norway,England,away team win,0.492974,0.211605,0.295420,away team win,True
548,48062,2026-07-14,France,Spain,away team win,0.482119,0.212638,0.305243,away team win,True
549,48063,2026-07-15,England,Argentina,away team win,0.444457,0.215289,0.340254,away team win,True
550,48064,2026-07-18,France,England,away team win,0.355532,0.215551,0.428917,home team win,False


In [37]:
predicciones_M2

,match_id,date,home_team,away_team,result,p_away,p_draw,p_home,prediccion,correcta
5323,47962,2026-06-11,South Korea,Czech Republic,home team win,0.394864,0.225399,0.379738,away team win,False
5324,47963,2026-06-11,Mexico,South Africa,home team win,0.069859,0.194858,0.735283,home team win,True
5325,47964,2026-06-12,Canada,Bosnia and Herzegovina,draw,0.105636,0.237142,0.657222,home team win,False
5326,47965,2026-06-12,United States,Paraguay,home team win,0.317203,0.275072,0.407725,home team win,True
5327,47966,2026-06-13,Australia,Turkey,home team win,0.510536,0.227713,0.261750,away team win,False
...,...,...,...,...,...,...,...,...,...,...
5422,48061,2026-07-11,Norway,England,away team win,0.545277,0.201026,0.253697,away team win,True
5423,48062,2026-07-14,France,Spain,away team win,0.501028,0.207400,0.291572,away team win,True
5424,48063,2026-07-15,England,Argentina,away team win,0.504767,0.223094,0.272138,away team win,True
5425,48064,2026-07-18,France,England,away team win,0.356690,0.220517,0.422792,home team win,False


In [38]:
predicciones_M3

,match_id,date,home_team,away_team,result,p_away,p_draw,p_home,prediccion,correcta
472,47986,2026-06-18,Switzerland,Bosnia and Herzegovina,home team win,0.207852,0.139240,0.652909,home team win,True
473,47987,2026-06-18,Canada,Qatar,home team win,0.046327,0.037403,0.916270,home team win,True
474,47988,2026-06-18,Czech Republic,South Africa,draw,0.146556,0.207956,0.645488,home team win,False
475,47989,2026-06-18,Mexico,South Korea,home team win,0.173640,0.058183,0.768177,home team win,True
476,47990,2026-06-19,United States,Australia,home team win,0.225981,0.076839,0.697179,home team win,True
...,...,...,...,...,...,...,...,...,...,...
547,48061,2026-07-11,Norway,England,away team win,0.433313,0.197798,0.368889,away team win,True
548,48062,2026-07-14,France,Spain,away team win,0.454892,0.195360,0.349749,away team win,True
549,48063,2026-07-15,England,Argentina,away team win,0.455812,0.165146,0.379042,away team win,True
550,48064,2026-07-18,France,England,away team win,0.349532,0.188824,0.461644,home team win,False


Para evaluar el desempeño de cada uno de estos modelos podemos utilizar la precision y el valor log loss de cada uno.  
Volveremos a reutilizar la siguiente funcion del notebook anterior para esto:  


In [39]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
def evaluar_modelo(predicciones):
    clases = ["away team win", "draw", "home team win"]

    y_real = predicciones["result"]
    y_pred = predicciones["prediccion"]

    accuracy = accuracy_score(y_real, y_pred)

    probabilidades = predicciones[
            ["p_away", "p_draw", "p_home"]
        ].to_numpy()

    perdida_log = log_loss(
        y_real,
        probabilidades,
        labels=clases
    )

    return {
        "accuracy": accuracy,
        "log_loss": perdida_log
    }



Recordemos que como el log loss es una suma de penalizaciones asignado cada uno a una probabilidad, este varia dependiendo del tamaño de muestra, y como el tamaño de prueba no es el mismo para todos los modelos ( M3 no puede utilizar los primeros 24 partidos del mundial porque tiene datos NaN respecto a estos), no podremos comparar directamente cada modelo sobre su respectiva serie de entrenamiento.  
Lo que si podemos hacer es comparar cada modelo sobre una misma serie de prueba, la cual podria ser la serie de prueba de M3, esto nos va a permitir comparar la capacidad predictiva de los modelos fuera de la muestra de entrenamiento.  

Para hacer esto, reduzcamos las predicciones de M1 y M2 a los 80 partidos que estan en las predicciones de M3:

In [40]:
predicciones_M1 = predicciones_M1.iloc[24:]
predicciones_M2 = predicciones_M2.iloc[24:]

In [41]:
predicciones_modelos = {
    "M1": predicciones_M1,
    "M2": predicciones_M2,
    "M3": predicciones_M3,
}

comparacion = pd.DataFrame({
    nombre: evaluar_modelo(predicciones)
    for nombre, predicciones in predicciones_modelos.items()
}).T

display(comparacion)

,accuracy,log_loss
M1,0.6875,0.765143
M2,0.7250,0.757510
M3,0.7375,0.724204


En esta serie de prueba, el modelo que mejor predice parece ser M3, tanto en precision como en las probabilidades que le asigna a los valores reales del partido (log_loss). Esto es consistente con el hecho de que tambien es el modelo que mejor se ajusta a la serie de entrenamiento.   
Veamos tambien como se ajustan estos modelos a la serie de prueba que utilizamos para comparar los modelos en el notebook previo:   


In [42]:
predicciones_M1 = predicciones_M1.iloc[-20:]
predicciones_M2 = predicciones_M2.iloc[-20:]
predicciones_M3 = predicciones_M3.iloc[-20:]

In [43]:
predicciones_modelos = {
    "M1": predicciones_M1,
    "M2": predicciones_M2,
    "M3": predicciones_M3,
}

comparacion = pd.DataFrame({
    nombre: evaluar_modelo(predicciones)
    for nombre, predicciones in predicciones_modelos.items()
}).T

display(comparacion)

,accuracy,log_loss
M1,0.7,0.723254
M2,0.8,0.705204
M3,0.9,0.621173


Sobre los mismos 20 partidos, M3 obtuvo un accuracy de 90%, M5a en el notebook pasado habia conseguido un 85%, pero esta diferencia solo equivale a un equipo correctamente clasificado, por lo que no nos permite demostrar que M3 sea mejor en prediccion.   
En el log-loss, M3 obtuvo uno de 0.62, superior al 0.45 de M5a, recordar que mayores valores de log-loss implican que el modelo asigna probabilidades menores a los resultados reales, y que la transformacion de probabilidades a penalizaciones es no lineal, es severa para unos pocos resultados con baja probabilidad.  

Veamos si la diferencia de los log-loss se deben a una mejora consistente asignada a las probabilidades reales en M5a o por unos pocos resultados reales en M3 asignados con bajas probabilidades, obteniendo una penalizacion exponencialmente mas alta y modificando el promedio.  

Nos podriamos dar cuenta de esto viendo la distribucion de las penalizaciones individuales en ambos modelos, esto nos puede permitir identificar si esta diferencia en el promedio se pudo haber dado por una mejor asignacion de probabilidades en M5a consistente entre los partidos, o por unos pocos casos en los que M3 asigno una probabilidad muy baja al resultado real.   

Para eso, obtengamos predicciones M5a:

In [ ]:
predicciones_M5a = pd.read_csv("predicciones_M5a.csv")

Y veamos la distribucion de sus probabilidades asignadas al resultado real y sus log-loss:

# ANIADIR PLOTS DE DISTRIBUCION DE PENALIZACIONES PARA AMBOS MODELOS CON HISTOGRAMAS, ANALIZAR 